# Импорты

In [ ]:
!pip install transformers==4.28.1
!pip install torch


In [ ]:
!pip -q install transformers torch sentencepiece

In [ ]:
!pip install rich.progress

# Загрузка модели и датасета

In [ ]:
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM
from typing import List, Dict, Any
from collections import defaultdict
import os

model_name = "FacebookAI/xlm-roberta-base"
# model_name = "google-bert/bert-base-multilingual-uncased"
print(f"Загрузка модели {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Модель загружена на {device}\n")

In [ ]:
with open("'/content/denoiseML/data/noise_data_banking_credit.json'", "r", encoding="utf-8") as f:
   banking_credit_noise_dataset = json.load(f)

# Предсказания

### Основные функции для MLM-разметки

In [ ]:
import json
import math
from typing import List, Dict, Any
from rich.progress import track
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

def join_words(words: List[str]) -> str:
    return " ".join(words)

def get_word_to_subword_mapping(words: List[str], tokenizer):
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        add_special_tokens=True
    )

    word_ids = encoding.word_ids(batch_index=0)

    word_to_token_positions = [[] for _ in range(len(words))]
    for token_pos, word_id in enumerate(word_ids):
        if word_id is not None:
            word_to_token_positions[word_id].append(token_pos)

    return encoding, word_to_token_positions


@torch.no_grad()
def mlm_word_scores(words: List[str], tokenizer, model, device="cpu") -> List[float]:
    encoding, word_to_token_positions = get_word_to_subword_mapping(words, tokenizer)

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    scores = []

    for word_idx, sub_positions in enumerate(word_to_token_positions):
        if not sub_positions:
            scores.append(0.0)
            continue

        masked_input_ids = input_ids.clone()
        original_token_ids = []

        for pos in sub_positions:
            original_token_ids.append(input_ids[0, pos].item())
            masked_input_ids[0, pos] = tokenizer.mask_token_id

        outputs = model(input_ids=masked_input_ids, attention_mask=attention_mask)
        logits = outputs.logits[0]

        subtoken_probs = []
        for pos, original_id in zip(sub_positions, original_token_ids):
            probs = torch.softmax(logits[pos], dim=-1)
            p = probs[original_id].item()
            subtoken_probs.append(max(p, 1e-12))

        log_mean = sum(math.log(p) for p in subtoken_probs) / len(subtoken_probs)
        word_score = math.exp(log_mean)
        scores.append(word_score)

    return scores

def predict_noise_labels_from_scores(
    scores: List[float],
    threshold: float = 1e-4,
    noise_label: str = "N",
    clean_label: str = "0"
) -> List[str]:
    return [noise_label if s < threshold else clean_label for s in scores]

def filter_words_by_pred(words: List[str], pred_labels: List[str], noise_label: str = "N") -> List[str]:
    return [w for w, lab in zip(words, pred_labels) if lab != noise_label]

def process_dataset(
    dataset: List[Dict[str, Any]],
    tokenizer,
    model,
    device="cpu",
    threshold: float = 1e-4
) -> List[Dict[str, Any]]:

    results = []

    for item in track(dataset, description="Разметка"):
        words = item["text"]
        gold_labels = item.get("label", None)

        scores = mlm_word_scores(words, tokenizer, model, device=device)
        pred_labels = predict_noise_labels_from_scores(scores, threshold=threshold)
        filtered_words = filter_words_by_pred(words, pred_labels)

        new_item = {
            **item,
            "mlm_score": [round(s, 8) for s in scores],
            "mlm_pred_label": pred_labels,
            "filtered_text": filtered_words,
            "filtered_text_str": " ".join(filtered_words)
        }

        if gold_labels is not None and len(gold_labels) == len(pred_labels):
            acc = sum(g == p for g, p in zip(gold_labels, pred_labels)) / len(gold_labels)
            new_item["mlm_token_accuracy"] = round(acc, 4)

        results.append(new_item)

    return results

### Подбор оптимального порога на маленькой выборке

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

small_data = banking_credit_noise_dataset[:10]
best = 0
best_threshold = 0.000000000000000000000000001

for threshold in [5e-1, 1e-1, 5e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5, 5e-6, 1e-6]:
  results = process_dataset(
      dataset=small_data,
      tokenizer=tokenizer,
      model=model,
      device=device,
      threshold=threshold
  )

  y_true = []
  y_pred = []
  correct_sentences = 0
  total_sentences = len(results)

  for item in results:
      y_true.extend(item["label"])
      y_pred.extend(item["mlm_pred_label"])
      if item["label"] == item["mlm_pred_label"]:
        correct_sentences += 1

  sentence_accuracy = correct_sentences / total_sentences if total_sentences > 0 else 0.0

  precision, recall, f1, support = precision_recall_fscore_support(
      y_true,
      y_pred,
      labels=["N"],
      average=None,
      zero_division=0
  )
  print(f"{threshold}:\t{sentence_accuracy=:.4f}, {f1[0]=}")
  if f1[0] > best:
    best = f1[0]
    best_threshold = threshold
  print(sum([1 if t==p else 0 for t, p in zip(y_true, y_pred)])/len(y_pred))
print(f"Best threschold: {best_threshold:.4f}")
print(f"Best f1: {best:.4f}")

### Обработка всего датасета с лучшим порогом

In [ ]:
results_all = process_dataset(
    dataset=banking_credit_noise_dataset,
    tokenizer=tokenizer,
    model=model,
    device=device,
    threshold=best_threshold
)
for i, item in enumerate(results, 1):
    print("=" * 100)
    print(f"EXAMPLE {i}")
    print("oldlabel:", item.get("oldlabel"))
    print("original text: ", " ".join(item["text"]))
    print("gold labels:   ", item.get("label", []))
    print("mlm scores:    ", item["mlm_score"])
    print("pred labels:   ", item["mlm_pred_label"])
    print("filtered text: ", item["filtered_text_str"])
    if "mlm_token_accuracy" in item:
        print("token accuracy:", item["mlm_token_accuracy"])

with open("mlm_filtered_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nSaved to mlm_filtered_results.json")

### Финальная оценка качества на всем датасете

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

y_true = []
y_pred = []
correct_sentences = 0
total_sentences = len(results_all)

for item in results_all:
    y_true.extend(item["label"])
    y_pred.extend(item["mlm_pred_label"])
    if item["label"] == item["mlm_pred_label"]:
      correct_sentences += 1

sentence_accuracy = correct_sentences / total_sentences if total_sentences > 0 else 0.0

print("Correct sentences:", correct_sentences)
print("Total sentences:  ", total_sentences)
print("Sentence accuracy:", round(sentence_accuracy, 4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=["N"],
    average=None,
    zero_division=0
)

print("Для класса 'N':")
print("Precision:", round(precision[0], 4))
print("Recall:   ", round(recall[0], 4))
print("F1-score: ", round(f1[0], 4))
print("Support:  ", int(support[0]))

print("\nПолный classification report:")
print(classification_report(y_true, y_pred, digits=4, zero_division=0))